# Generate All Charts

Produces charts 01–21 as HTML + PNG into `Final/charts/{descriptive,cluster,regression,ml}/`.

Run top-to-bottom. Charts that depend on missing CSVs are skipped gracefully.


In [ ]:
# ---------------------------------------------------------------------------
# 0. Walk up to project root (contains 'intermediary/' and 'Final/')
# ---------------------------------------------------------------------------
import os, sys

def _find_root(marker='intermediary'):
    try:
        d = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        # __file__ not defined (e.g., Jupyter)
        d = os.getcwd()

    for _ in range(6):
        if os.path.isdir(os.path.join(d, marker)):
            return d
        d = os.path.dirname(d)
    raise RuntimeError(f"Could not find project root (looking for '{marker}' dir).")

ROOT = _find_root()

os.chdir(ROOT)
sys.path.insert(0, os.path.join(ROOT, 'scripts'))

# ---------------------------------------------------------------------------
# 1. Standard library + third-party imports
# ---------------------------------------------------------------------------
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ---------------------------------------------------------------------------
# 2. Import shared utilities from viz_utils
# ---------------------------------------------------------------------------
from viz_utils import (
    PALETTE, FONT, BG, NAVY, GRID, WRITE_CONFIG,
    base_layout, save,
    load_master, load_master_wide, load_clusters, load_nr, load_nb5, load_bootstrap,
    INCLUDE_LIST, resource_rich_codes, build_sample, shorten_feat,
    CLUSTER_LABELS, CLUSTER_COLORS, LABEL_TO_COLOR,
    analyze_country_missingness,
)

# ---------------------------------------------------------------------------
# 3. Output directory
# ---------------------------------------------------------------------------
OUT_DESCR = os.path.join(ROOT, 'Final', 'charts', 'descriptive')
OUT_CLUST = os.path.join(ROOT, 'Final', 'charts', 'cluster')
OUT_REGR  = os.path.join(ROOT, 'Final', 'charts', 'regression')
OUT_ML    = os.path.join(ROOT, 'Final', 'charts', 'ml')
for _d in [OUT_DESCR, OUT_CLUST, OUT_REGR, OUT_ML]:
    os.makedirs(_d, exist_ok=True)

# ---------------------------------------------------------------------------
# 4. Helper: hex → rgb tuple (used in chart 11)
# ---------------------------------------------------------------------------
def _hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))


# ---------------------------------------------------------------------------
# 5. Clustering pipeline (re-runs PCA + KMeans; used for charts 03, 04a/b/c/d)
# ---------------------------------------------------------------------------
_LABEL_COLORS_4K = {
    'Petrostates':      '#d4853b',
    'Oil Exporters':    '#4a6fa5',
    'Major Producers':  '#2e7d4a',
    'Forestry Intensive':'#c23a3a',   # red
}

_LABEL_COLORS_4F = {
    'Petrostates':       '#d4853b',
    'Oil Exporters':     '#c23a3a',
    'Oil & Minerals':    '#7a5c9e',
    'Mineral Exporters': '#2e7d4a',
    'Low Resource':      '#4a6fa5',
}

_LABEL_COLORS_5K = {
    'Petrostates':        '#d4853b',   # orange
    'Oil Exporters':      '#4a6fa5',   # blue
    'Diversified Producers': '#2e7d4a',   # green
    'Forestry Intensive': '#c23a3a',   # red
    'Mining Exporters':   '#7a5c9e',   # purple
}

_LABEL_COLORS_6K = {
    'Petrostates':           '#d4853b',   # orange
    'Oil Exporters':         '#4a6fa5',   # blue
    'Forestry Intensive':    '#c23a3a',   # red
    'Oil & Minerals':        '#3a8fa5',   # teal
    'Mining Exporters':      '#7a5c9e',   # purple
    'Diversified Producers': '#2e7d4a',   # green
}


def run_clustering(nr_data, year_filter=None, agg_years=None, n_clusters=4, random_state=42, label_override=None):
    """Full pipeline: pivot → per-capita → log1p → PCA(2) → KMeans(k)."""
    df = nr_data.copy()
    if year_filter is not None:
        df = df[df['Year'] == year_filter]
    elif agg_years is not None:
        df = df[df['Year'].isin(agg_years)]

    df_pivot = df.pivot_table(
        index=['Country', 'Country Code', 'Year', 'Population'],
        columns='Resource', values='Production_TotalValue',
    ).reset_index()

    resource_cols = df_pivot.columns.difference(['Country', 'Country Code', 'Year', 'Population'])
    df_pivot[resource_cols] = df_pivot[resource_cols].div(df_pivot['Population'], axis=0)
    df_pivot.drop(columns='Population', inplace=True)
    df_pivot = df_pivot.fillna(0)

    df_latest = (df_pivot.sort_values('Year', ascending=True)
                 .groupby(['Country', 'Country Code']).first().reset_index())

    feature_cols = [c for c in df_latest.columns if c not in ['Country', 'Country Code', 'Year']]
    X_log = np.log1p(df_latest[feature_cols].fillna(0))

    pca = PCA(n_components=2)
    pca_components = pca.fit_transform(X_log)

    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state)
    clusters = kmeans.fit_predict(pca_components)

    pca_df = pd.DataFrame({
        'Country':      df_latest['Country'],
        'Country Code': df_latest['Country Code'],
        'Year':         df_latest['Year'],
        'PC1': pca_components[:, 0],
        'PC2': pca_components[:, 1],
        'Cluster': clusters,
    })

    centroids = kmeans.cluster_centers_
    pc1_rank  = list(np.argsort(-centroids[:, 0]))
    pc2_rank  = list(np.argsort(-centroids[:, 1]))

    if label_override is not None:
        label_map = label_override
    else:
        label_map, labeled = {}, set()
        oil_id = pc1_rank[0]
        label_map[oil_id] = 'Petrostates'; labeled.add(oil_id)
        mineral_id = next(c for c in pc2_rank if c not in labeled)
        label_map[mineral_id] = 'Major Producers'; labeled.add(mineral_id)
        remaining = [c for c in pc1_rank if c not in labeled]
        label_map[remaining[0]] = 'Oil Exporters'
        label_map[remaining[1]] = 'Forestry Intensive'

    pca_df['ClusterLabels'] = pca_df['Cluster'].map(label_map)
    return pca_df, pca, feature_cols


def run_clustering_4feat(nr_data, year_filter=None, n_clusters=5, random_state=42):
    """Cluster on Oil / Natural Gas / Coal / Minerals (aggregate), k=5."""
    df = nr_data.copy()
    if year_filter is not None:
        df = df[df['Year'] == year_filter]

    KEEP = ['Oil', 'Natural Gas', 'Coal']
    df['_Category'] = df['Resource'].apply(lambda r: r if r in KEEP else 'Minerals')
    df_agg = (df.groupby(['Country', 'Country Code', 'Year', 'Population', '_Category'])
              ['Production_TotalValue'].sum().reset_index())

    pivot = df_agg.pivot_table(
        index=['Country', 'Country Code', 'Year', 'Population'],
        columns='_Category', values='Production_TotalValue',
    ).reset_index().fillna(0)

    feat_cols = [c for c in ['Coal', 'Minerals', 'Natural Gas', 'Oil'] if c in pivot.columns]
    pivot[feat_cols] = pivot[feat_cols].div(pivot['Population'], axis=0)
    pivot = pivot.fillna(0)

    X = np.log1p(pivot[feat_cols])
    pca = PCA(n_components=2)
    Xp  = pca.fit_transform(X)

    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state)
    labels = km.fit_predict(Xp)

    pca_df = pd.DataFrame({
        'Country':      pivot['Country'],
        'Country Code': pivot['Country Code'],
        'Year':         pivot['Year'],
        'PC1': Xp[:, 0], 'PC2': Xp[:, 1],
        'Cluster': labels,
    })

    centroids = km.cluster_centers_
    pc1_rank  = list(np.argsort(-centroids[:, 0]))
    pc2_rank  = list(np.argsort(-centroids[:, 1]))

    label_map, labeled = {}, set()
    label_map[pc1_rank[0]] = 'Petrostates'; labeled.add(pc1_rank[0])
    min_id = next(c for c in pc2_rank if c not in labeled)
    label_map[min_id] = 'Mineral Exporters'; labeled.add(min_id)
    remaining = [c for c in pc1_rank if c not in labeled]
    label_map[remaining[0]] = 'Oil & Minerals'
    label_map[remaining[1]] = 'Oil Exporters'
    label_map[remaining[2]] = 'Low Resource'

    pca_df['ClusterLabels'] = pca_df['Cluster'].map(label_map)
    return pca_df


def create_cluster_map(pca_df, nr_data, cluster_names_map=None,
                       label_colors=None, dominance_threshold=15.0):
    """Choropleth map with red borders for major global producers."""
    if cluster_names_map is None:
        cluster_names_map = dict(zip(pca_df['Cluster'].unique(), pca_df['ClusterLabels'].unique()))
    if label_colors is None:
        label_colors = _LABEL_COLORS_4K

    df_total = nr_data.pivot_table(
        index=['Country', 'Country Code'],
        columns='Resource', values='Production_TotalValue', aggfunc='sum',
    ).reset_index().fillna(0)

    prod_cols  = [c for c in df_total.columns if c not in ['Country', 'Country Code']]
    for col in prod_cols:
        total = df_total[col].sum()
        if total > 0:
            df_total[f'{col}_Share'] = (df_total[col] / total) * 100

    share_cols = [c for c in df_total.columns if c.endswith('_Share')]
    df_map = pca_df.merge(df_total[['Country Code'] + share_cols], on='Country Code', how='left')
    df_map['Is_Dominant']       = (df_map[share_cols] >= dominance_threshold).any(axis=1)
    df_map['Dominant_Resources'] = df_map.apply(
        lambda row: [sc.replace('_Share', '') for sc in share_cols
                     if row.get(sc, 0) >= dominance_threshold], axis=1)

    def make_hover(row):
        lbl   = row['ClusterLabels']
        lines = [f"<b>{row['Country']}</b>", f"Cluster: {lbl}"]
        vals  = [(c, row.get(c, 0)) for c in prod_cols if row.get(c, 0) > 0]
        vals.sort(key=lambda x: x[1], reverse=True)
        if vals:
            lines.append('<br>Top Resources:')
            for res, v in vals[:3]:
                if v > 1e9:   lines.append(f'  {res}: ${v/1e9:.1f}B')
                elif v > 1e6: lines.append(f'  {res}: ${v/1e6:.0f}M')
                else:         lines.append(f'  {res}: ${v:,.0f}')
        return '<br>'.join(lines)

    df_map['hover_text'] = df_map.apply(make_hover, axis=1)

    fig = go.Figure()
    for cid in sorted(df_map['Cluster'].unique()):
        lbl   = cluster_names_map.get(cid, f'Cluster {cid}')
        color = label_colors.get(lbl, '#aaa')

        sub = df_map[(df_map['Cluster'] == cid) & (~df_map['Is_Dominant'])]
        if len(sub) > 0:
            fig.add_trace(go.Choropleth(
                locations=sub['Country Code'], z=[cid]*len(sub),
                colorscale=[[0, color], [1, color]], showscale=False,
                showlegend=True, name=lbl,
                customdata=sub['hover_text'].values,
                hovertemplate='%{customdata}<extra></extra>',
                marker=dict(line=dict(color='white', width=0.6)),
            ))

        sub_d = df_map[(df_map['Cluster'] == cid) & (df_map['Is_Dominant'])]
        if len(sub_d) > 0:
            fig.add_trace(go.Choropleth(
                locations=sub_d['Country Code'], z=[cid]*len(sub_d),
                colorscale=[[0, color], [1, color]], showscale=False,
                showlegend=False, name=f'{lbl} ★ major producer',
                customdata=sub_d['hover_text'].values,
                hovertemplate='%{customdata}<extra></extra>',
                marker=dict(line=dict(color='#111', width=2.2)),
            ))

    fig.add_trace(go.Choropleth(
        locations=['ZZZ'], z=[0],
        colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']],
        showscale=False, showlegend=True,
        name='★ >15% of global output',
        marker=dict(line=dict(color='#111', width=2.2)),
    ))

    fig.update_geos(
        projection_type='natural earth',
        showcountries=True, countrycolor='#ccc',
        showcoastlines=True, coastlinecolor='#ccc',
        showland=True, landcolor='#f0f0f0',
        showocean=True, oceancolor='#dde8f0',
        showframe=False,
    )
    fig.update_layout(
        margin=dict(l=0, r=0, t=50, b=70),
        legend=dict(
            orientation='h', x=0.5, y=-0.08, xanchor='center', yanchor='top',
            font=dict(size=11, family=FONT),
            bgcolor='rgba(250,250,250,0.9)', bordercolor='#d0d0d0', borderwidth=1,
        ),
        paper_bgcolor=BG, plot_bgcolor=BG,
        font=dict(family=FONT),
    )
    return fig


# ============================================================================
# Pre-run: build cluster data (needed by charts 01, 03, 04a/b/c/d, 05, 26)
# ============================================================================
print('Loading NaturalResource.csv and running clustering pipeline...')
nr_full   = load_nr()
nr_sample = nr_full[nr_full['Country Code'].isin(INCLUDE_LIST)]

pca_1995, pca_model_1995, feat_1995 = run_clustering(nr_sample, year_filter=1995)
pca_2019, pca_model_2019, feat_2019 = run_clustering(nr_sample, year_filter=2019)
pca_agg,  pca_model_agg,  feat_agg  = run_clustering(nr_sample, agg_years=[1995, 1999, 2005])

# k=5 and k=6 variants — 1995 only
# Cluster IDs are stable under random_state=42; verified from centroid inspection.
_K5_LABELS = {
    0: 'Forestry Intensive',
    1: 'Petrostates',
    2: 'Mining Exporters',
    3: 'Oil Exporters',
    4: 'Diversified Producers',
}
_K6_LABELS = {
    0: 'Petrostates',
    1: 'Forestry Intensive',
    2: 'Oil & Minerals',
    3: 'Oil Exporters',
    4: 'Mining Exporters',
    5: 'Diversified Producers',
}
pca_1995_k5, _, _ = run_clustering(nr_sample, year_filter=1995, n_clusters=5, label_override=_K5_LABELS)
pca_1995_k6, _, _ = run_clustering(nr_sample, year_filter=1995, n_clusters=6, label_override=_K6_LABELS)
print('  Clustering done.')


# =============================================================================

Loading NaturalResource.csv and running clustering pipeline...
  Clustering done.


---
# Descriptive Statistics  ·  Charts 01–02


## CHART 01 — Variable correlations with ECI (Pearson, panel 1995–2019)


In [2]:
# =============================================================================
print('\n=== CHART 01 ===')

master = load_master()
panel  = build_sample(master)

# Variables grouped by category (matching reference chart)
FEAT_CATEGORIES = {
    # Governance
    'Political stability — estimate':                                  'Governance',
    'Rule of law index':                                                'Governance',
    'Property rights':                                                  'Governance',
    'Political corruption index':                                       'Governance',
    # Human Capital & Infrastructure
    'Human capital index':                                              'Human Capital & Infrastructure',
    'Life expectancy at birth, total (years)':                         'Human Capital & Infrastructure',
    'Access to electricity (% of population)':                         'Human Capital & Infrastructure',
    'Mobile cellular subscriptions (per 100 people)':                  'Human Capital & Infrastructure',
    'Urban population (% of total population)':                        'Human Capital & Infrastructure',
    # Finance & Investment
    'Domestic credit to private sector (% of GDP)':                    'Finance & Investment',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP': 'Finance & Investment',
    'Adjusted savings: gross savings (% of GNI)':                     'Finance & Investment',
    'Capital depreciation rate':                                        'Finance & Investment',
    'Lending interest rate (%)':                                        'Finance & Investment',
    # Macro & Structure
    'GDP per capita (constant prices, PPP)':                           'Macro & Structure',
    'Manufacturing, value added (% of GDP)':                           'Macro & Structure',
    'Services, value added (% of GDP)':                                'Macro & Structure',
    'Government revenue':                                               'Macro & Structure',
    'Trade (% of GDP)':                                                 'Macro & Structure',
    'Industry (including construction), value added (% of GDP)':       'Macro & Structure',
    'Inflation, consumer prices (annual %)':                            'Macro & Structure',
    'Agriculture, forestry, and fishing, value added (% of GDP)':     'Macro & Structure',
    # Resource Rents
    'Natural gas rents (% of GDP)':                                     'Resource Rents',
    'Oil rents (% of GDP)':                                             'Resource Rents',
    'Mineral rents (% of GDP)':                                         'Resource Rents',
    'Forest rents (% of GDP)':                                          'Resource Rents',
    'Total natural resources rents (% of GDP)':                        'Resource Rents',
}

CAT_COLORS = {
    'Resource Rents':                  '#c23a3a',
    'Macro & Structure':               '#7a5c9e',
    'Finance & Investment':            '#d4a017',
    'Human Capital & Infrastructure':  '#2e7d4a',
    'Governance':                      '#4a6fa5',
}

# Group display order: bottom to top on the y-axis
GROUP_ORDER = ['Resource Rents', 'Macro & Structure', 'Finance & Investment',
               'Human Capital & Infrastructure', 'Governance']

eci_col = 'Economic Complexity Index'
corr_rows = []
for col, cat in FEAT_CATEGORIES.items():
    if col not in panel.columns:
        continue
    sub = panel[[eci_col, col]].dropna()
    if len(sub) < 20:
        continue
    r = sub[eci_col].corr(sub[col])
    corr_rows.append({'Feature': col, 'Correlation': r, 'Category': cat})

corr_df = pd.DataFrame(corr_rows)
corr_df['Label'] = corr_df['Feature'].apply(shorten_feat)

# Sort: within each group by correlation; groups in GROUP_ORDER (first group at bottom)
corr_df['_grp_rank'] = corr_df['Category'].map({g: i for i, g in enumerate(GROUP_ORDER)})
corr_df = corr_df.sort_values(['_grp_rank', 'Correlation'], ascending=[False, True]).reset_index(drop=True)
ordered_labels = corr_df['Label'].tolist()

# One trace per category for the legend
fig02 = go.Figure()
for cat in GROUP_ORDER:
    sub = corr_df[corr_df['Category'] == cat]
    if sub.empty:
        continue
    fig02.add_trace(go.Bar(
        x=sub['Correlation'], y=sub['Label'],
        orientation='h', name=cat,
        marker=dict(color=CAT_COLORS[cat], opacity=0.85,
                    line=dict(color='white', width=0.5)),
        hovertemplate='%{y}: %{x:.3f}<extra></extra>',
    ))

fig02.add_vline(x=0, line=dict(color='#444', width=1.5))
fig02.update_layout(**base_layout(
    height=max(500, len(corr_df) * 22),
    margin=dict(l=180, r=80, t=60, b=60),
    xaxis=dict(title='Pearson r with ECI',
               gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(tickfont=dict(size=10), categoryorder='array',
               categoryarray=ordered_labels),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, title_text='Category',
                font=dict(size=10)),
    barmode='overlay',
))
save(fig02, '01_descr__variable_correlations_with_eci', OUT_DESCR, w=1100, h=700)


# =============================================================================



=== CHART 01 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/descriptive/01_descr__variable_correlations_with_eci.png


## CHART 02 — ECI median trajectory by cluster


In [3]:
# =============================================================================
print('\n=== CHART 02 ===')

master15   = load_master()
k5_assign  = pca_1995_k5[['Country Code', 'Cluster', 'ClusterLabels']].drop_duplicates()
df15       = master15[master15['Country Code'].isin(INCLUDE_LIST)].copy()
df15       = df15.merge(k5_assign, on='Country Code', how='left')

traj15 = (df15.groupby(['Year', 'Cluster', 'ClusterLabels'])['Economic Complexity Index']
          .median().reset_index())

fig15 = go.Figure()
for cl in sorted(traj15['Cluster'].dropna().unique()):
    sub = traj15[traj15['Cluster'] == cl]
    lbl = sub['ClusterLabels'].iloc[0]
    fig15.add_trace(go.Scatter(
        x=sub['Year'], y=sub['Economic Complexity Index'],
        mode='lines+markers',
        name=lbl,
        line=dict(color=_LABEL_COLORS_5K.get(lbl, '#999'), width=2.2),
        marker=dict(size=5),
        hovertemplate='%{x}: %{y:.3f}<extra>' + lbl + '</extra>',
    ))

fig15.update_layout(**base_layout(
    height=480,
    xaxis=dict(title='Year', gridcolor=GRID, gridwidth=0.5, dtick=5),
    yaxis=dict(title='Median ECI', gridcolor=GRID, gridwidth=0.5,
               zeroline=True, zerolinecolor='#ddd', zerolinewidth=1),
    legend=dict(font=dict(size=10), bgcolor='rgba(255,255,255,0.9)',
                bordercolor=GRID, borderwidth=1),
    hovermode='x unified',
))
save(fig15, '02_descr__eci_mean_trajectory_by_cluster', OUT_DESCR, w=1100, h=480)


# =============================================================================


=== CHART 02 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/descriptive/02_descr__eci_mean_trajectory_by_cluster.png


## CHART 02b — Cluster profiles: normalised variable means, k=5

In [4]:
# =============================================================================
print('\n=== CHART 02b ===')

# Variables, display names, and category groups
_PROFILE_VARS = {
    'Resource Rents': [
        ('Total natural resources rents (% of GDP)', 'NR Rents'),
        ('Oil rents (% of GDP)',                     'Oil Rents'),
    ],
    'Macro & Structure': [
        ('GDP per capita (constant prices, PPP)',     'GDP per capita'),
        ('Agriculture',                               'Agriculture'),
    ],
    'Finance & Invest.': [
        ('Domestic credit to private sector (% of GDP)',                     'Domestic Credit'),
        ('Gross fixed capital formation, all, Constant prices, Percent of GDP', 'GFCF'),
    ],
    'Human Capital': [
        ('Human capital index',                          'Human Capital'),
        ('Life expectancy at birth, total (years)',       'Life Expectancy'),
    ],
    'Governance': [
        ('Rule of law index',           'Rule of Law'),
        ('Political corruption index',  'Pol. Corruption'),
    ],
}

_GROUP_COLORS = {
    'Resource Rents':    '#c23a3a',
    'Macro & Structure': '#4a6fa5',
    'Finance & Invest.': '#d4853b',
    'Human Capital':     '#2e7d4a',
    'Governance':        '#7a5c9e',
}

# Build 1995 sample merged with k=5 clusters
master_p   = load_master()
master_p95 = master_p[(master_p['Year'] == 1995) &
                       (master_p['Country Code'].isin(INCLUDE_LIST))].copy()
k5_assign  = pca_1995_k5[['Country Code', 'ClusterLabels']].drop_duplicates()
master_p95 = master_p95.merge(k5_assign, on='Country Code', how='left')

# Ordered cluster list for consistent bar order
_K5_ORDER = ['Petrostates', 'Oil Exporters', 'Diversified Producers',
             'Forestry Intensive', 'Mining Exporters']

# Flatten vars and compute normalised cluster means
all_cols  = [col for grp in _PROFILE_VARS.values() for col, _ in grp]
all_short = [srt for grp in _PROFILE_VARS.values() for _, srt in grp]

profile = master_p95.groupby('ClusterLabels')[all_cols].mean()

# Min-max normalise across the full sample (not just cluster means)
sample_means = master_p95[all_cols].mean()
vmin = master_p95[all_cols].min()
vmax = master_p95[all_cols].max()
norm = (profile - vmin) / (vmax - vmin).replace(0, 1)

fig_p = go.Figure()

bar_x = all_short   # x-axis tick labels

for lbl in _K5_ORDER:
    if lbl not in norm.index:
        continue
    fig_p.add_trace(go.Bar(
        x=bar_x,
        y=[norm.loc[lbl, col] for col, _ in
           [pair for grp in _PROFILE_VARS.values() for pair in grp]],
        name=lbl,
        marker_color=_LABEL_COLORS_5K.get(lbl, '#aaa'),
        opacity=0.88,
    ))

# Category header annotations
x_positions = []
pos = 0
for grp, pairs in _PROFILE_VARS.items():
    n = len(pairs)
    mid = pos + (n - 1) / 2
    x_positions.append((grp, mid, pos, pos + n - 1))
    pos += n

fig_p.update_layout(**base_layout(
    height=440,
    barmode='group',
    bargap=0.18,
    bargroupgap=0.05,
    margin=dict(l=60, r=40, t=80, b=90),
    xaxis=dict(tickangle=-35, tickfont=dict(size=10),
               showgrid=False, zeroline=False),
    yaxis=dict(title='Normalised mean (0 = sample min, 1 = sample max)',
               range=[0, 1.05], gridcolor=GRID, gridwidth=0.5,
               tickfont=dict(size=10)),
    legend=dict(orientation='h', x=0.5, y=-0.28, xanchor='center',
                font=dict(size=10), bgcolor='rgba(255,255,255,0)'),
))

# Category group labels — coloured bold text only, no underlines
W, L, R = 1200, 60, 40
plot_w = 1 - L/W - R/W
l_frac = L / W
n_vars = len(bar_x)
var_w  = plot_w / n_vars

pos = 0
for grp, pairs in _PROFILE_VARS.items():
    n    = len(pairs)
    col  = _GROUP_COLORS[grp]
    xc_p = l_frac + (pos + n / 2) * var_w
    fig_p.add_annotation(
        x=xc_p, y=1.07, xref='paper', yref='paper',
        text=f'<b>{grp}</b>', showarrow=False,
        font=dict(size=11, color=col, family=FONT),
        xanchor='center', yanchor='bottom',
    )
    pos += n

save(fig_p, '02b_cluster__profile_normalised_means_k5', OUT_CLUST, w=1200, h=440)


# =============================================================================


=== CHART 02b ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/cluster/02b_cluster__profile_normalised_means_k5.png


---
# Cluster Analysis  ·  Charts 03–11


## CHART 03 — PCA Biplot (1995): country scatter + resource arrows


In [5]:
# =============================================================================
print('\n=== CHART 03 ===')

def _ellipse_points(x_vals, y_vals, n_std=2.5, n_pts=120):
    # Covariance ellipse scaled to encompass the full cluster spread.
    pts = np.column_stack([x_vals, y_vals])
    if len(pts) < 3:
        mu = pts.mean(axis=0)
        t  = np.linspace(0, 2 * np.pi, n_pts + 1)
        return mu[0] + 0.7 * np.cos(t), mu[1] + 0.7 * np.sin(t)
    mu             = pts.mean(axis=0)
    cov            = np.cov(pts.T)
    eigvals, vecs  = np.linalg.eigh(cov)
    eigvals        = np.maximum(eigvals, 0)
    t              = np.linspace(0, 2 * np.pi, n_pts + 1)
    circle         = np.column_stack([np.cos(t), np.sin(t)])
    ellipse        = n_std * (vecs @ (np.diag(np.sqrt(eigvals)) @ circle.T)).T + mu
    return ellipse[:, 0], ellipse[:, 1]

def _radial_textpos(px, py, cx, cy):
    """Return a Plotly textposition string based on angle from cluster centroid."""
    angle = np.degrees(np.arctan2(py - cy, px - cx))
    if   angle < -157.5 or angle >= 157.5: return 'middle left'
    elif angle < -112.5:                   return 'bottom left'
    elif angle <  -67.5:                   return 'bottom center'
    elif angle <  -22.5:                   return 'bottom right'
    elif angle <   22.5:                   return 'middle right'
    elif angle <   67.5:                   return 'top right'
    elif angle <  112.5:                   return 'top center'
    else:                                  return 'top left'

def chart_03_biplot(pca_df, pca_model, feature_cols):
    loadings_df = pd.DataFrame(
        pca_model.components_.T * np.sqrt(pca_model.explained_variance_),
        columns=['PC1', 'PC2'], index=feature_cols,
    )
    importance = loadings_df.abs().sum(axis=1)
    top5  = importance.nlargest(5).index
    top10 = importance.nlargest(10).index
    scale = 2.8

    var1 = pca_model.explained_variance_ratio_[0] * 100
    var2 = pca_model.explained_variance_ratio_[1] * 100

    fig = go.Figure()

    # ── ellipses first (render under markers) ────────────────────────────────
    _nstd = {
        'Forestry Intensive':    2.9,   # wide spread incl. ZWE/ZMB/VNM
        'Diversified Producers': 1.7,   # compact enough — 6 countries but far apart
    }
    for cid in sorted(pca_df['Cluster'].unique()):
        sub   = pca_df[pca_df['Cluster'] == cid]
        lbl   = sub['ClusterLabels'].iloc[0]
        color = _LABEL_COLORS_5K.get(lbl, '#999')
        r, g, b = _hex_to_rgb(color)
        ex, ey  = _ellipse_points(sub['PC1'].values, sub['PC2'].values,
                                   n_std=_nstd.get(lbl, 2.5))
        fig.add_trace(go.Scatter(
            x=ex, y=ey,
            mode='lines',
            line=dict(color=f'rgba({r},{g},{b},0.55)', width=1.6, dash='dot'),
            fill='toself',
            fillcolor=f'rgba({r},{g},{b},0.07)',
            showlegend=False,
            hoverinfo='skip',
        ))

    # ── scatter markers ───────────────────────────────────────────────────────
    for cid in sorted(pca_df['Cluster'].unique()):
        sub   = pca_df[pca_df['Cluster'] == cid]
        lbl   = sub['ClusterLabels'].iloc[0]
        color = _LABEL_COLORS_5K.get(lbl, '#999')
        cx, cy = sub['PC1'].mean(), sub['PC2'].mean()
        tpos   = [_radial_textpos(r['PC1'], r['PC2'], cx, cy)
                  for _, r in sub.iterrows()]
        fig.add_trace(go.Scatter(
            x=sub['PC1'], y=sub['PC2'],
            mode='markers+text',
            marker=dict(size=10, color=color, opacity=0.82,
                        line=dict(width=1.2, color='white')),
            text=sub['Country Code'],
            textposition=tpos,
            textfont=dict(size=8, color='#333'),
            name=lbl,
            hovertemplate='<b>%{text}</b><br>PC1=%{x:.2f}, PC2=%{y:.2f}<extra></extra>',
        ))

    # ── loading arrows ────────────────────────────────────────────────────────
    for feat_name in top10:
        if feat_name in top5:
            continue
        fig.add_annotation(
            x=loadings_df.loc[feat_name, 'PC1'] * scale,
            y=loadings_df.loc[feat_name, 'PC2'] * scale,
            ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=2, arrowsize=0.8,
            arrowwidth=1.2, arrowcolor='rgba(150,150,150,0.5)',
        )

    for feat_name in top5:
        x1 = loadings_df.loc[feat_name, 'PC1'] * scale
        y1 = loadings_df.loc[feat_name, 'PC2'] * scale
        fig.add_annotation(
            x=x1, y=y1, ax=0, ay=0,
            xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=3, arrowsize=1.2,
            arrowwidth=2.2, arrowcolor='#222',
        )
        fig.add_annotation(
            x=x1 * 1.18, y=y1 * 1.18,
            text=f'<b>{feat_name}</b>', showarrow=False,
            font=dict(size=10, color='#111', family=FONT),
            bgcolor='rgba(255,255,255,0.7)', borderpad=2,
        )

    fig.add_hline(y=0, line=dict(color=GRID, width=1))
    fig.add_vline(x=0, line=dict(color=GRID, width=1))

    fig.update_layout(**base_layout(
        height=680,
        margin=dict(l=60, r=60, t=50, b=60),
        xaxis=dict(title=f'PC1 ({var1:.1f}% variance explained)',
                   gridcolor=GRID, gridwidth=0.5),
        yaxis=dict(title=f'PC2 ({var2:.1f}% variance explained)',
                   gridcolor=GRID, gridwidth=0.5),
        legend=dict(title='Resource profile (1995)', font=dict(size=10),
                    bgcolor='rgba(250,250,250,0.85)',
                    bordercolor=GRID, borderwidth=1),
    ))
    return fig

fig03 = chart_03_biplot(pca_1995_k5, pca_model_1995, feat_1995)
save(fig03, '03_cluster__pca_biplot_country_resource_groups', OUT_CLUST, w=1100, h=680)


# =============================================================================



=== CHART 03 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/cluster/03_cluster__pca_biplot_country_resource_groups.png


## CHART 04 — PCA Resource Loadings Heatmap


In [6]:
# =============================================================================
print('\n=== CHART 04 ===')

nr26      = load_nr()
nr_s26    = nr26[nr26['Country Code'].isin(INCLUDE_LIST)]
nr_1995_26 = nr_s26[nr_s26['Year'] == 1995]

pivot26 = nr_1995_26.pivot_table(
    index=['Country', 'Country Code', 'Year', 'Population'],
    columns='Resource', values='Production_TotalValue',
).reset_index()

resource_cols26 = [c for c in pivot26.columns
                   if c not in ['Country', 'Country Code', 'Year', 'Population']]
pivot26[resource_cols26] = pivot26[resource_cols26].div(pivot26['Population'], axis=0)
pivot26 = pivot26.fillna(0)

X26  = np.log1p(pivot26[resource_cols26].fillna(0))
pca26 = PCA(n_components=2, random_state=42)
pca26.fit(X26)

var1_26 = pca26.explained_variance_ratio_[0] * 100
var2_26 = pca26.explained_variance_ratio_[1] * 100

loadings26 = pd.DataFrame(pca26.components_.T, columns=['PC1', 'PC2'],
                          index=resource_cols26)
top20_26   = loadings26.abs().sum(axis=1).nlargest(20).index
plot_df26  = loadings26.loc[top20_26]
plot_df26  = (plot_df26.assign(_s=plot_df26['PC1'].abs() + plot_df26['PC2'].abs())
              .sort_values('_s', ascending=False).drop(columns='_s'))

pc_labels_ordered = [
    f'PC1 ({var1_26:.1f}%)<br><i>↑ Oil & Gas</i>',
    f'PC2 ({var2_26:.1f}%)<br><i>↑ Copper, Gold & Coal</i>',
]
y_labels26 = pc_labels_ordered[::-1]
z_values26 = plot_df26[['PC2', 'PC1']].T.values

fig26 = go.Figure(go.Heatmap(
    z=z_values26,
    x=plot_df26.index.tolist(),
    y=y_labels26,
    colorscale=[[0.0, '#1a4a8a'], [0.5, '#ffffff'], [1.0, '#c23a3a']],
    zmid=0, zmin=-1, zmax=1,
    hovertemplate='<b>%{x}</b><br>%{y}: %{z:.3f}<extra></extra>',
    colorbar=dict(
        title=dict(text='Loading', font=dict(size=12)),
        thickness=20, len=1.0,
        tickvals=[-1, -0.5, 0, 0.5, 1],
        tickfont=dict(size=11),
    ),
))

fig26.update_xaxes(title_text='Resource/Feature', tickangle=-40,
                   tickfont=dict(size=10, family=FONT), showgrid=False)
fig26.update_yaxes(title_text='Principal Component',
                   tickfont=dict(size=12, family=FONT), showgrid=False)
fig26.update_layout(**base_layout(
    height=420, margin=dict(l=260, r=120, t=60, b=160),
))
save(fig26, '04_cluster__pca_resource_loadings_heatmap', OUT_CLUST, w=1300, h=420)


# =============================================================================


=== CHART 04 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/cluster/04_cluster__pca_resource_loadings_heatmap.png


## CHART 05 — Cluster world map, 1995 snapshot


In [7]:
# =============================================================================
print('\n=== CHART 05 ===')

nr_1995_sub   = nr_sample[nr_sample['Year'] == 1995]
cnames_1995   = dict(zip(pca_1995['Cluster'], pca_1995['ClusterLabels']))
fig04a = create_cluster_map(pca_1995, nr_1995_sub, cluster_names_map=cnames_1995)
fig04a.update_layout(height=520)
save(fig04a, '05_cluster__world_map_1995_resource_profiles', OUT_CLUST, w=1200, h=520)


# =============================================================================


=== CHART 05 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/cluster/05_cluster__world_map_1995_resource_profiles.png


## CHART 06 — Cluster world map, 1995, k=5

In [8]:
# =============================================================================
print('\n=== CHART 06 ===')

nr_1995_sub     = nr_sample[nr_sample['Year'] == 1995]
cnames_1995_k5  = dict(zip(pca_1995_k5['Cluster'], pca_1995_k5['ClusterLabels']))
fig09b = create_cluster_map(pca_1995_k5, nr_1995_sub,
                             cluster_names_map=cnames_1995_k5,
                             label_colors=_LABEL_COLORS_5K)
fig09b.update_layout(height=520)
save(fig09b, '06_cluster__world_map_1995_k5', OUT_CLUST, w=1200, h=520)


# =============================================================================


=== CHART 06 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/cluster/06_cluster__world_map_1995_k5.png


## CHART 07 — Cluster world map, 1995, k=6

In [9]:
# =============================================================================
print('\n=== CHART 07 ===')

nr_1995_sub     = nr_sample[nr_sample['Year'] == 1995]
cnames_1995_k6  = dict(zip(pca_1995_k6['Cluster'], pca_1995_k6['ClusterLabels']))
fig09c = create_cluster_map(pca_1995_k6, nr_1995_sub,
                             cluster_names_map=cnames_1995_k6,
                             label_colors=_LABEL_COLORS_6K)
fig09c.update_layout(height=520)
save(fig09c, '07_cluster__world_map_1995_k6', OUT_CLUST, w=1200, h=520)


# =============================================================================


=== CHART 07 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/cluster/07_cluster__world_map_1995_k6.png


## CHART 08 — Cluster world map, 2019 snapshot


In [10]:
# =============================================================================
print('\n=== CHART 08 ===')

nr_2019_sub = nr_sample[nr_sample['Year'] == 2019]
cnames_2019 = dict(zip(pca_2019['Cluster'], pca_2019['ClusterLabels']))
fig04b = create_cluster_map(pca_2019, nr_2019_sub, cluster_names_map=cnames_2019)
fig04b.update_layout(height=520)
save(fig04b, '08_cluster__world_map_2019_resource_profiles', OUT_CLUST, w=1200, h=520)


# =============================================================================


=== CHART 08 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/cluster/08_cluster__world_map_2019_resource_profiles.png


## CHART 09 — Cluster world map, aggregated (1995/1999/2005)


In [11]:
# =============================================================================
print('\n=== CHART 09 ===')

nr_agg_sub = nr_sample[nr_sample['Year'].isin([1995, 1999, 2005])]
cnames_agg = dict(zip(pca_agg['Cluster'], pca_agg['ClusterLabels']))
fig04c = create_cluster_map(pca_agg, nr_agg_sub, cluster_names_map=cnames_agg)
fig04c.update_layout(height=520)
save(fig04c, '09_cluster__world_map_agg_resource_profiles', OUT_CLUST, w=1200, h=520)


# =============================================================================


=== CHART 09 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/cluster/09_cluster__world_map_agg_resource_profiles.png


## CHART 10 — 4-feature map (Oil / Gas / Coal / Minerals), k=5


In [12]:
# =============================================================================
print('\n=== CHART 10 ===')

pca_4f   = run_clustering_4feat(nr_sample, year_filter=1995)
cnames_4f = dict(zip(pca_4f['Cluster'], pca_4f['ClusterLabels']))
fig04d = create_cluster_map(pca_4f, nr_1995_sub, cluster_names_map=cnames_4f,
                            label_colors=_LABEL_COLORS_4F)
fig04d.update_layout(height=520)
save(fig04d, '10_cluster__world_map_4feat_oil_gas_coal_minerals', OUT_CLUST, w=1200, h=520)


# =============================================================================


=== CHART 10 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/cluster/10_cluster__world_map_4feat_oil_gas_coal_minerals.png


## CHART 11 — Animated Rosling chart: ECI vs log(GDP pc), 1995–2019


In [13]:
# =============================================================================
print('\n=== CHART 11 ===')

def chart_05_rosling(master_df, pca_df_agg, cluster_colors, cluster_names,
                     arrow_opacity=0.5, arrow_width=2):
    """Animated ECI vs log(GDP pc) Rosling chart with trajectory arrows."""
    data = master_df.copy()
    data['Log GDP per capita'] = np.log(
        data['GDP per capita (constant prices, PPP)'].replace(0, np.nan))
    data['Production_Per_Capita'] = (data['Total_Production_Value']
                                     / data['Population'].replace(0, np.nan))

    c1995 = (data[data['Year'] == 1995][['Country Code', 'Cluster']]
             .rename(columns={'Cluster': 'Cluster_1995'}))
    data  = data.merge(c1995, on='Country Code', how='left')
    data  = data.dropna(subset=['Cluster_1995', 'Log GDP per capita',
                                 'Economic Complexity Index', 'Production_Per_Capita'])
    data['Cluster_1995'] = data['Cluster_1995'].astype(int)

    data['Bubble_Size'] = np.sqrt(data['Production_Per_Capita'])
    mn, mx = data['Bubble_Size'].min(), data['Bubble_Size'].max()
    data['Bubble_Size_Scaled'] = 8 + (data['Bubble_Size'] - mn) / (mx - mn) * 42

    data  = data.sort_values(['Year', 'Country Code'])
    years = sorted(data['Year'].unique())
    countries_list = data['Country Code'].unique()
    clusters_all   = sorted(data['Cluster_1995'].unique())

    cdata = {}
    for code in countries_list:
        cdf    = data[data['Country Code'] == code].sort_values('Year')
        origin = cdf[cdf['Year'] == 1995]
        if len(origin) == 0:
            continue
        cdata[code] = {
            'years': cdf['Year'].values,
            'x': cdf['Log GDP per capita'].values,
            'y': cdf['Economic Complexity Index'].values,
            'x0': origin['Log GDP per capita'].values[0],
            'y0': origin['Economic Complexity Index'].values[0],
            'size': cdf['Bubble_Size_Scaled'].values,
            'name': cdf['Country Name'].iloc[0],
            'cluster': int(cdf['Cluster_1995'].iloc[0]),
            'prod_pc': cdf['Production_Per_Capita'].values,
        }

    valid_countries = list(cdata.keys())
    first_year = years[0]

    fig = go.Figure()

    for cl in clusters_all:
        cc    = [c for c in valid_countries if cdata[c]['cluster'] == cl]
        color = cluster_colors.get(cl, '#999999')

        for code in cc:
            cd  = cdata[code]
            idx = np.where(cd['years'] == first_year)[0]
            xc  = cd['x'][idx[0]] if len(idx) > 0 else cd['x0']
            yc  = cd['y'][idx[0]] if len(idx) > 0 else cd['y0']
            fig.add_trace(go.Scatter(
                x=[cd['x0'], xc], y=[cd['y0'], yc],
                mode='lines', line=dict(color=color, width=arrow_width),
                opacity=arrow_opacity, legendgroup=f'cl_{cl}',
                showlegend=False, hoverinfo='skip',
            ))

        for code in cc:
            cd  = cdata[code]
            idx = np.where(cd['years'] == first_year)[0]
            if len(idx) > 0:
                i = idx[0]
                xv, yv, sv, pv = [cd['x'][i]], [cd['y'][i]], cd['size'][i], cd['prod_pc'][i]
            else:
                xv, yv, sv, pv = [cd['x0']], [cd['y0']], 15, 0
            fig.add_trace(go.Scatter(
                x=xv, y=yv, mode='markers+text',
                marker=dict(size=sv, color=color, opacity=0.85,
                            line=dict(width=1.5, color='white')),
                text=[code], textposition='top center',
                textfont=dict(size=8, color='black'),
                name=cluster_names.get(cl, f'Cluster {cl}'),
                legendgroup=f'cl_{cl}', showlegend=(code == cc[0]),
                customdata=[[cd['name'], pv, first_year]],
                hovertemplate='<b>%{customdata[0]}</b><br>Log GDP pc: %{x:.2f}<br>'
                              'ECI: %{y:.2f}<br>Prod/capita: $%{customdata[1]:,.0f}<br>'
                              'Year: %{customdata[2]}<extra></extra>',
            ))

        for code in cc:
            cd = cdata[code]
            fig.add_trace(go.Scatter(
                x=[cd['x0']], y=[cd['y0']], mode='markers',
                marker=dict(size=5, color=color, opacity=0.6, symbol='circle'),
                legendgroup=f'cl_{cl}', showlegend=False, hoverinfo='skip',
            ))

    frames = []
    for year in years:
        fd = []
        for cl in clusters_all:
            cc    = [c for c in valid_countries if cdata[c]['cluster'] == cl]
            color = cluster_colors.get(cl, '#999999')
            for code in cc:
                cd  = cdata[code]
                idx = np.where(cd['years'] == year)[0]
                if len(idx) > 0:
                    xc, yc = cd['x'][idx[0]], cd['y'][idx[0]]
                else:
                    mask = cd['years'] <= year
                    li   = np.where(mask)[0][-1] if mask.any() else 0
                    xc, yc = cd['x'][li], cd['y'][li]
                fd.append(go.Scatter(x=[cd['x0'], xc], y=[cd['y0'], yc],
                                     mode='lines', line=dict(color=color, width=arrow_width),
                                     opacity=arrow_opacity))
            for code in cc:
                cd  = cdata[code]
                idx = np.where(cd['years'] == year)[0]
                if len(idx) > 0:
                    i = idx[0]
                    xv, yv, sv, pv = [cd['x'][i]], [cd['y'][i]], cd['size'][i], cd['prod_pc'][i]
                else:
                    mask = cd['years'] <= year
                    if mask.any():
                        li = np.where(mask)[0][-1]
                        xv, yv, sv, pv = [cd['x'][li]], [cd['y'][li]], cd['size'][li], cd['prod_pc'][li]
                    else:
                        xv, yv, sv, pv = [cd['x0']], [cd['y0']], 15, 0
                fd.append(go.Scatter(
                    x=xv, y=yv, mode='markers+text',
                    marker=dict(size=sv, color=color, opacity=0.85,
                                line=dict(width=1.5, color='white')),
                    text=[code], textposition='top center',
                    textfont=dict(size=8),
                    customdata=[[cd['name'], pv, year]],
                    hovertemplate='<b>%{customdata[0]}</b><br>Log GDP pc: %{x:.2f}<br>'
                                  'ECI: %{y:.2f}<br>Prod/capita: $%{customdata[1]:,.0f}<br>'
                                  'Year: %{customdata[2]}<extra></extra>',
                ))
            for code in cc:
                cd = cdata[code]
                fd.append(go.Scatter(x=[cd['x0']], y=[cd['y0']], mode='markers',
                                     marker=dict(size=5, color=color, opacity=0.6,
                                                 symbol='circle')))
        frames.append(go.Frame(data=fd, name=str(year)))

    fig.frames = frames

    eci_vals = data['Economic Complexity Index']
    x_vals   = data['Log GDP per capita']
    fig.update_layout(
        xaxis=dict(range=[x_vals.min()-0.2, x_vals.max()+0.2],
                   title='Log GDP per capita (PPP, constant 2017 USD)'),
        yaxis=dict(range=[eci_vals.min()-0.15, eci_vals.max()+0.15],
                   title='Economic Complexity Index'),
        plot_bgcolor=BG, paper_bgcolor=BG,
        font=dict(family=FONT, color=NAVY),
        legend=dict(title='Resource profile (1995 cluster)', x=1.02, y=0.99),
        sliders=[dict(
            active=0, len=0.85, x=0.05, y=-0.12,
            currentvalue=dict(prefix='Year: ', font=dict(size=14)),
            steps=[dict(
                args=[[str(y)], dict(frame=dict(duration=300, redraw=True), mode='immediate')],
                method='animate', label=str(y),
            ) for y in years],
        )],
    )
    return fig


master_ros = load_master()
master_ros = master_ros[master_ros['Country Code'].isin(INCLUDE_LIST)].copy()
master_ros = master_ros.merge(
    pca_agg[['Country Code', 'Cluster', 'ClusterLabels']],
    on='Country Code', how='left',
)

CLUSTER_COLORS_AGG = {
    cid: _LABEL_COLORS_4K.get(
        pca_agg.loc[pca_agg['Cluster'] == cid, 'ClusterLabels'].iloc[0], '#aaa')
    for cid in sorted(pca_agg['Cluster'].unique())
}
CLUSTER_NAMES_AGG = dict(zip(pca_agg['Cluster'], pca_agg['ClusterLabels']))

fig05 = chart_05_rosling(master_ros, pca_agg, CLUSTER_COLORS_AGG, CLUSTER_NAMES_AGG)
out_html = os.path.join(OUT_CLUST, '11_cluster__eci_vs_gdp_animated_1995_to_2019.html')
fig05.write_html(out_html, config=WRITE_CONFIG)
print(f'  Saved: {out_html}')


# =============================================================================


=== CHART 11 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/cluster/11_cluster__eci_vs_gdp_animated_1995_to_2019.html


---
# Machine Learning  ·  Charts 12–21


## CHART 12 — Feature Importance Consensus (LASSO / Ridge / Elastic Net)


In [14]:
# =============================================================================
print('\n=== CHART 12 ===')

_imp_path = os.path.join('Final', 'NB5', 'all_importance.csv')
if os.path.exists(_imp_path):
    LABEL_EXCL = ['L1_ECI', 'Inflation_roll5', 'RealRate_roll5', 'Resource_HHI']
    imp = pd.read_csv(_imp_path)
    imp = imp[~imp['Feature'].apply(lambda f: any(e in f for e in LABEL_EXCL))]

    lin_cols = [c for c in ['LASSO', 'Ridge', 'Elastic Net'] if c in imp.columns]
    sort_col = 'Elastic Net' if 'Elastic Net' in imp.columns else lin_cols[0]
    imp = (imp.sort_values(sort_col, ascending=False)
             .head(12).reset_index(drop=True))
    imp = imp.iloc[::-1].reset_index(drop=True)
    imp['Label'] = imp['Feature'].apply(shorten_feat)

    fig07 = go.Figure()
    for _, row in imp.iterrows():
        vals = [row[c] for c in lin_cols if not pd.isna(row[c])]
        if len(vals) >= 2:
            fig07.add_trace(go.Scatter(
                x=[min(vals), max(vals)], y=[row['Label'], row['Label']],
                mode='lines', line=dict(color='#c0c8d4', width=3),
                showlegend=False, hoverinfo='skip',
            ))

    model_cfg07 = [
        ('LASSO',       'circle',      PALETTE['lasso']),
        ('Ridge',       'square',      PALETTE['ridge']),
        ('Elastic Net', 'triangle-up', PALETTE['en']),
    ]
    for mname, sym, col in model_cfg07:
        if mname not in imp.columns:
            continue
        fig07.add_trace(go.Scatter(
            x=imp[mname], y=imp['Label'],
            mode='markers',
            marker=dict(symbol=sym, size=13, color=col,
                        line=dict(color='white', width=1.5)),
            name=mname,
            hovertemplate=f'%{{y}}: %{{x:.3f}}<extra>{mname}</extra>',
        ))

    x_max = imp[lin_cols].max().max()
    fig07.update_layout(**base_layout(
        height=560,
        margin=dict(l=200, r=80, t=70, b=80),
        xaxis=dict(title='Normalised Feature Importance (min-max, 0–1)',
                   range=[-0.02, x_max + 0.1], gridcolor=GRID, gridwidth=0.5),
        yaxis=dict(tickfont=dict(size=11)),
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='center', x=0.5, font=dict(size=11)),
    ))
    save(fig07, '12_ml__feature_importance_consensus_three_models', OUT_ML, w=1100, h=560)
else:
    print('  SKIPPED: all_importance.csv not found')


# =============================================================================


=== CHART 12 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/ml/12_ml__feature_importance_consensus_three_models.png


## CHART 13 — Standardised Coefficients (LASSO / Ridge / Elastic Net)


In [15]:
# =============================================================================
print('\n=== CHART 13 ===')

_tbl_path = os.path.join('Final', 'NB5', 'coefficient_summary_table.csv')
if os.path.exists(_tbl_path):
    tbl = pd.read_csv(_tbl_path)
    LABEL_EXCL = ['L1_ECI', 'Inflation_roll5', 'RealRate_roll5', 'Resource_HHI']
    tbl = tbl[~tbl['Feature'].apply(lambda f: any(e in f for e in LABEL_EXCL))]
    tbl['abs_en'] = tbl['Elastic Net'].abs()
    top = tbl.nlargest(12, 'abs_en').sort_values('abs_en', ascending=True).reset_index(drop=True)

    fig08 = go.Figure()
    fig08.add_vline(x=0, line=dict(color='#444', width=1.5))

    model_cfg08 = [
        ('LASSO',       PALETTE['lasso']),
        ('Ridge',       PALETTE['ridge']),
        ('Elastic Net', PALETTE['en']),
    ]
    for mname, col in model_cfg08:
        if mname not in top.columns:
            continue
        fig08.add_trace(go.Bar(
            y=top['Feature'], x=top[mname], orientation='h',
            name=mname,
            marker=dict(color=col, opacity=0.88, line=dict(color='white', width=0.5)),
            hovertemplate=f'%{{y}}: %{{x:+.3f}}<extra>{mname}</extra>',
        ))

    fig08.update_layout(**base_layout(
        barmode='group', height=620,
        margin=dict(l=200, r=80, t=70, b=60),
        xaxis=dict(title='Coefficient (standardised inputs)',
                   gridcolor=GRID, gridwidth=0.5, zeroline=False),
        yaxis=dict(tickfont=dict(size=11)),
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='center', x=0.5, font=dict(size=11)),
    ))
    save(fig08, '13_ml__standardised_coefficients_lasso_ridge_en', OUT_ML, w=1100, h=620)
else:
    print('  SKIPPED: coefficient_summary_table.csv not found')


# =============================================================================


=== CHART 13 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/ml/13_ml__standardised_coefficients_lasso_ridge_en.png


## CHART 14 — Random Forest Feature Importance


In [16]:
# =============================================================================
print('\n=== CHART 14 ===')
_imp_path = os.path.join('Final', 'NB5', 'all_importance.csv')
LABEL_EXCL = ['L1_ECI', 'Inflation_roll5', 'RealRate_roll5', 'Resource_HHI']

if os.path.exists(_imp_path):
    imp27 = pd.read_csv(_imp_path)
    imp27 = imp27[imp27['Model'] == 'RandomForest'] if 'Model' in imp27.columns else imp27.copy()

    if 'Model' not in imp27.columns:
        # all_importance.csv has one row per feature, RF column exists
        rf_col = 'Random Forest' if 'Random Forest' in imp27.columns else None
        if rf_col:
            imp27 = (imp27[['Feature', rf_col]]
                     .rename(columns={rf_col: 'Importance'})
                     .dropna())
    else:
        imp27 = imp27.rename(columns={'Importance': 'Importance'}) if 'Importance' in imp27.columns else imp27

    # Fallback: use the RF column from all_importance.csv
    if 'Importance' not in imp27.columns:
        rf_col = next((c for c in imp27.columns if 'Forest' in c or 'RF' in c or 'rf' in c.lower()), None)
        if rf_col:
            imp27 = imp27[['Feature', rf_col]].rename(columns={rf_col: 'Importance'}).dropna()

    if 'Importance' in imp27.columns and 'Feature' in imp27.columns:
        imp27 = imp27[~imp27['Feature'].apply(
            lambda f: any(e in f for e in LABEL_EXCL))].copy()
        imp27 = (imp27.sort_values('Importance', ascending=False)
                 .head(15).iloc[::-1].reset_index(drop=True))
        imp27['Label'] = imp27['Feature'].apply(shorten_feat)

        fig27 = go.Figure(go.Bar(
            x=imp27['Importance'], y=imp27['Label'], orientation='h',
            marker=dict(color=PALETTE['rf'], opacity=0.88,
                        line=dict(color='white', width=0.5)),
            hovertemplate='%{y}: %{x:.3f}<extra>Random Forest</extra>',
        ))
        fig27.update_layout(**base_layout(
            height=540,
            margin=dict(l=200, r=80, t=60, b=60),
            xaxis=dict(title='Feature Importance', gridcolor=GRID, gridwidth=0.5),
            yaxis=dict(tickfont=dict(size=11)),
        ))
        save(fig27, '14_ml__random_forest_feature_importance', OUT_ML, w=1100, h=540)
    else:
        print('  SKIPPED: could not extract RF importance column')
else:
    print('  SKIPPED: all_importance.csv not found')


# =============================================================================


=== CHART 14 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/ml/14_ml__random_forest_feature_importance.png


## CHART 15 — Train vs Test R² (horizontal bars)


In [17]:
# =============================================================================
print('\n=== CHART 15 ===')

_perf_l_path = os.path.join('Final', 'NB5', 'model_performance_level.csv')
_perf_d_path = os.path.join('Final', 'NB5', 'model_performance_delta.csv')
if os.path.exists(_perf_l_path) and os.path.exists(_perf_d_path):
    perf_l = pd.read_csv(_perf_l_path)
    perf_d = pd.read_csv(_perf_d_path)
    perf_l = perf_l[perf_l['Model'] != 'XGBoost'].reset_index(drop=True)
    perf_d = perf_d[perf_d['Model'] != 'XGBoost'].reset_index(drop=True)

    fig09 = make_subplots(rows=1, cols=2, horizontal_spacing=0.14)

    for col_idx, (perf, panel_lbl) in enumerate([(perf_l, 'ECI Level'), (perf_d, 'ΔECI')], 1):
        models   = perf['Model'].tolist()
        train_r2 = perf['Train R²'].tolist()
        test_r2  = perf['Test R²'].tolist()

        for m, tr, te in zip(models, train_r2, test_r2):
            fig09.add_trace(go.Scatter(
                x=[tr, te], y=[m, m], mode='lines',
                line=dict(color='#c0c8d4', width=2.5),
                showlegend=False, hoverinfo='skip',
            ), row=1, col=col_idx)

        fig09.add_trace(go.Scatter(
            x=train_r2, y=models, mode='markers',
            marker=dict(symbol='circle', size=13, color=PALETTE['blue'],
                        line=dict(color='white', width=1.5)),
            name='Train R²', showlegend=(col_idx == 1), legendgroup='train',
            hovertemplate='%{y} Train: %{x:.3f}<extra></extra>',
        ), row=1, col=col_idx)

        fig09.add_trace(go.Scatter(
            x=test_r2, y=models, mode='markers',
            marker=dict(symbol='diamond', size=13, color=PALETTE['red'],
                        line=dict(color='white', width=1.5)),
            name='Test R²', showlegend=(col_idx == 1), legendgroup='test',
            hovertemplate='%{y} Test: %{x:.3f}<extra></extra>',
        ), row=1, col=col_idx)

        fig09.update_xaxes(title_text='R²', gridcolor=GRID, gridwidth=0.5,
                           row=1, col=col_idx)
        fig09.update_yaxes(tickfont=dict(size=11), row=1, col=col_idx)

    for x_paper, lbl in [(0.23, 'ECI Level'), (0.77, 'ΔECI')]:
        fig09.add_annotation(
            x=x_paper, y=1.04, xref='paper', yref='paper',
            text=f'<b>{lbl}</b>', showarrow=False,
            font=dict(size=12, color=NAVY, family=FONT),
            xanchor='center', yanchor='bottom',
        )

    fig09.update_layout(**base_layout(
        height=440, margin=dict(l=130, r=60, t=80, b=60),
        legend=dict(orientation='h', yanchor='bottom', y=1.06,
                    xanchor='center', x=0.5, font=dict(size=11)),
    ))
    save(fig09, '15_ml__train_vs_test_r2_horizontal', OUT_ML, w=1100, h=440)
else:
    print('  SKIPPED: model_performance CSVs not found')


# =============================================================================



=== CHART 15 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/ml/15_ml__train_vs_test_r2_horizontal.png


## CHART 16 — Train vs Test R² (vertical bars)


In [18]:
# =============================================================================
print('\n=== CHART 16 ===')

_perf_l_path = os.path.join('Final', 'NB5', 'model_performance_level.csv')
_perf_d_path = os.path.join('Final', 'NB5', 'model_performance_delta.csv')
if os.path.exists(_perf_l_path) and os.path.exists(_perf_d_path):
    perf_l = pd.read_csv(_perf_l_path)
    perf_d = pd.read_csv(_perf_d_path)
    perf_l = perf_l[perf_l['Model'] != 'XGBoost'].reset_index(drop=True)
    perf_d = perf_d[perf_d['Model'] != 'XGBoost'].reset_index(drop=True)

    fig09 = make_subplots(rows=1, cols=2, horizontal_spacing=0.14)

    for col_idx, (perf, panel_lbl) in enumerate([(perf_l, 'ECI Level'), (perf_d, 'ΔECI')], 1):
        models   = perf['Model'].tolist()
        train_r2 = perf['Train R²'].tolist()
        test_r2  = perf['Test R²'].tolist()

        fig09.add_trace(go.Bar(
            x=models, y=train_r2,
            marker=dict(color=PALETTE['blue'], line=dict(color='white', width=1)),
            name='Train R²', showlegend=(col_idx == 1), legendgroup='train',
            hovertemplate='%{x} Train: %{y:.3f}<extra></extra>',
        ), row=1, col=col_idx)

        fig09.add_trace(go.Bar(
            x=models, y=test_r2,
            marker=dict(color=PALETTE['red'], line=dict(color='white', width=1)),
            name='Test R²', showlegend=(col_idx == 1), legendgroup='test',
            hovertemplate='%{x} Test: %{y:.3f}<extra></extra>',
        ), row=1, col=col_idx)

        fig09.update_yaxes(title_text='R²', gridcolor=GRID, gridwidth=0.5,
                           row=1, col=col_idx)
        fig09.update_xaxes(tickfont=dict(size=11), tickangle=-30, row=1, col=col_idx)

    for x_paper, lbl in [(0.23, 'ECI Level'), (0.77, 'ΔECI')]:
        fig09.add_annotation(
            x=x_paper, y=1.04, xref='paper', yref='paper',
            text=f'<b>{lbl}</b>', showarrow=False,
            font=dict(size=12, color=NAVY, family=FONT),
            xanchor='center', yanchor='bottom',
        )

    fig09.update_layout(
        barmode='group',
        **base_layout(
            height=440, margin=dict(l=60, r=60, t=80, b=100),
            legend=dict(orientation='h', yanchor='bottom', y=1.06,
                        xanchor='center', x=0.5, font=dict(size=11)),
        ),
    )
    save(fig09, '16_ml__train_vs_test_r2_vertical', OUT_ML, w=1100, h=440)
else:
    print('  SKIPPED: model_performance CSVs not found')


=== CHART 16 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/ml/16_ml__train_vs_test_r2_vertical.png


## CHART 17 — Actual vs Predicted ECI (level + ΔECI, side by side)


In [19]:
# =============================================================================
print('\n=== CHART 17 ===')

_preds_path = os.path.join('Final', 'NB5', 'test_predictions.csv')
if os.path.exists(_preds_path):
    preds = pd.read_csv(_preds_path)

    fig10 = make_subplots(rows=1, cols=2, horizontal_spacing=0.12)

    for col_idx, (actual_col, pred_col, lbl) in enumerate([
        ('Actual_ECI',   'Predicted_ECI',   'ECI'),
        ('Actual_Delta', 'Predicted_Delta', 'ΔECI'),
    ], 1):
        if actual_col not in preds.columns or pred_col not in preds.columns:
            continue
        actual = preds[actual_col].dropna().values
        pred   = preds.loc[preds[actual_col].notna(), pred_col].values
        codes  = preds.loc[preds[actual_col].notna(), 'Country Code'].values
        names  = preds.loc[preds[actual_col].notna(), 'Country Name'].values

        lims = [min(actual.min(), pred.min()) - 0.1,
                max(actual.max(), pred.max()) + 0.1]
        mid  = 0.0

        for x0, x1, y0, y1, fc in [
            (lims[0], mid,     lims[0], mid,     'rgba(46,125,74,0.07)'),
            (mid,     lims[1], mid,     lims[1], 'rgba(46,125,74,0.07)'),
            (lims[0], mid,     mid,     lims[1], 'rgba(194,58,58,0.07)'),
            (mid,     lims[1], lims[0], mid,     'rgba(194,58,58,0.07)'),
        ]:
            fig10.add_shape(type='rect', x0=x0, x1=x1, y0=y0, y1=y1,
                            fillcolor=fc, line=dict(width=0), layer='below',
                            row=1, col=col_idx)

        fig10.add_trace(go.Scatter(
            x=[lims[0], lims[1]], y=[lims[0], lims[1]],
            mode='lines', line=dict(color=PALETTE['red'], width=1.5, dash='dash'),
            name='45° line', showlegend=(col_idx == 1), legendgroup='line45',
        ), row=1, col=col_idx)

        resid   = np.abs(actual - pred)
        top_idx = set(np.argsort(resid)[::-1][:5])
        mask_n  = np.array([i not in top_idx for i in range(len(actual))])

        fig10.add_trace(go.Scatter(
            x=actual[mask_n], y=pred[mask_n], mode='markers',
            marker=dict(size=6, color=PALETTE['blue'], opacity=0.65,
                        line=dict(color='white', width=0.5)),
            name='Test obs.', showlegend=(col_idx == 1), legendgroup='obs',
            customdata=np.stack([codes[mask_n], names[mask_n]], axis=1),
            hovertemplate='<b>%{customdata[1]}</b><br>'
                          'Actual: %{x:.3f}<br>Predicted: %{y:.3f}<extra></extra>',
        ), row=1, col=col_idx)

        out_idx = list(top_idx)
        fig10.add_trace(go.Scatter(
            x=actual[out_idx], y=pred[out_idx], mode='markers+text',
            marker=dict(size=9, color=PALETTE['orange'], opacity=0.9,
                        line=dict(color='white', width=1)),
            text=codes[out_idx], textposition='top center', textfont=dict(size=9),
            name='Largest residuals', showlegend=(col_idx == 1), legendgroup='outliers',
            customdata=np.stack([codes[out_idx], names[out_idx]], axis=1),
            hovertemplate='<b>%{customdata[1]}</b><br>'
                          'Actual: %{x:.3f}<br>Predicted: %{y:.3f}<extra></extra>',
        ), row=1, col=col_idx)

        fig10.add_hline(y=0, line=dict(color=GRID, width=1), row=1, col=col_idx)
        fig10.add_vline(x=0, line=dict(color=GRID, width=1), row=1, col=col_idx)
        fig10.update_xaxes(title_text=f'Actual {lbl} (test set)', range=lims,
                           gridcolor=GRID, gridwidth=0.5, row=1, col=col_idx)
        fig10.update_yaxes(title_text=f'Predicted {lbl}', range=lims,
                           gridcolor=GRID, gridwidth=0.5, row=1, col=col_idx)

    fig10.update_layout(**base_layout(
        height=560, margin=dict(l=70, r=50, t=70, b=60),
        legend=dict(orientation='h', yanchor='bottom', y=1.04,
                    xanchor='center', x=0.5, font=dict(size=10)),
    ))
    save(fig10, '17_ml__actual_vs_predicted_eci_test_set', OUT_ML, w=1100, h=560)
else:
    print('  SKIPPED: test_predictions.csv not found')


# =============================================================================


=== CHART 17 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/ml/17_ml__actual_vs_predicted_eci_test_set.png


## CHART 18 — ML Prediction Intervals (aggregated by country)


In [20]:
# =============================================================================
print('\n=== CHART 18 ===')

if os.path.exists(_preds_path):
    preds31 = pd.read_csv(_preds_path)

    country_stats = (
        preds31.groupby(['Country Code', 'Country Name'])
        .agg(
            Actual_mean    = ('Actual_ECI', 'mean'),
            Predicted_mean = ('Predicted_ECI', 'mean'),
            Actual_std     = ('Actual_ECI', 'std'),
            n_years        = ('Year', 'count'),
        )
        .reset_index()
        .sort_values('Actual_mean')
        .reset_index(drop=True)
    )
    country_stats['Actual_std'] = country_stats['Actual_std'].fillna(0)
    country_stats['In_band'] = (
        (country_stats['Actual_mean'] - country_stats['Predicted_mean']).abs()
        < country_stats['Actual_std']
    )

    fig31 = go.Figure()

    fig31.add_trace(go.Scatter(
        x=list(range(len(country_stats))) * 2
          + list(range(len(country_stats)))[::-1] * 2,
        y=(country_stats['Actual_mean'] + country_stats['Actual_std']).tolist()
          + (country_stats['Actual_mean'] - country_stats['Actual_std']).iloc[::-1].tolist(),
        fill='toself', fillcolor='rgba(74,111,165,0.15)',
        line=dict(color='rgba(0,0,0,0)'),
        hoverinfo='skip', name='±1 SD (actual ECI in test years)',
    ))

    fig31.add_trace(go.Scatter(
        x=list(range(len(country_stats))),
        y=country_stats['Actual_mean'],
        mode='lines', line=dict(color=PALETTE['blue'], width=2),
        name='Mean Actual ECI',
    ))

    for in_band, color, sym, lbl in [
        (True,  PALETTE['green'], 'circle',  'Predicted ≈ Actual (within ±1 SD)'),
        (False, PALETTE['red'],   'diamond', 'Predicted outside ±1 SD'),
    ]:
        mask = country_stats['In_band'] == in_band
        sub  = country_stats[mask]
        fig31.add_trace(go.Scatter(
            x=sub.index.tolist(),
            y=sub['Predicted_mean'],
            mode='markers',
            marker=dict(color=color, size=8 if in_band else 10,
                        symbol=sym, opacity=0.85),
            name=lbl,
            customdata=sub[['Country Code', 'Country Name']].values,
            hovertemplate='<b>%{customdata[1]}</b> (%{customdata[0]})<br>'
                          'Avg Actual: %{text}<br>Avg Predicted: %{y:.3f}<extra></extra>',
            text=[f'{v:.3f}' for v in sub['Actual_mean']],
        ))

    fig31.update_layout(**base_layout(
        height=500,
        xaxis=dict(
            title='Countries (sorted by mean actual ECI)',
            tickvals=list(range(len(country_stats))),
            ticktext=country_stats['Country Code'].tolist(),
            tickangle=-60, tickfont=dict(size=8),
            gridcolor=GRID,
        ),
        yaxis=dict(title='ECI (test set mean)', gridcolor=GRID, gridwidth=0.5),
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='center', x=0.5, font=dict(size=10)),
    ))
    save(fig31, '18_ml__ml_prediction_intervals', OUT_ML, w=1200, h=500)
else:
    print('  SKIPPED: test_predictions.csv not found')


# =============================================================================


=== CHART 18 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/ml/18_ml__ml_prediction_intervals.png


## CHART 19 — Bootstrap R² stability (ML models)


In [21]:
# =============================================================================
print('\n=== CHART 19 ===')

_boot_metrics = os.path.join('intermediary', 'bootstrap', 'nb5_boot_metrics.csv')
if os.path.exists(_boot_metrics):
    boot = pd.read_csv(_boot_metrics)

    model_cols = [c for c in boot.columns if c.endswith('_test_r2')]
    if model_cols:
        fig29 = go.Figure()
        colors29 = [PALETTE['lasso'], PALETTE['ridge'], PALETTE['en'], PALETTE['rf'],
                    PALETTE['teal'], PALETTE['purple']]

        for i, col in enumerate(model_cols):
            model_name = col.replace('_test_r2', '').replace('_', ' ')
            if 'XGBoost' in model_name:
                continue
            vals = boot[col].dropna()
            fig29.add_trace(go.Box(
                y=vals, name=model_name,
                marker=dict(color=colors29[i % len(colors29)], opacity=0.8),
                line=dict(color=colors29[i % len(colors29)]),
                boxmean='sd',
                hovertemplate=f'{model_name}<br>R²: %{{y:.3f}}<extra></extra>',
            ))

        fig29.update_layout(**base_layout(
            height=500,
            margin=dict(l=80, r=60, t=60, b=80),
            xaxis=dict(title='Model', gridcolor=GRID),
            yaxis=dict(title='Bootstrap Test R²', gridcolor=GRID, gridwidth=0.5),
        ))
        save(fig29, '19_ml__bootstrap_r2_stability', OUT_ML, w=1100, h=500)
    else:
        print('  SKIPPED: no *_test_r2 columns in bootstrap metrics')
else:
    print('  SKIPPED: nb5_boot_metrics.csv not found')


# =============================================================================


=== CHART 19 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/ml/19_ml__bootstrap_r2_stability.png


## CHART 20 — ECI Forecast: best improvers vs worst decliners (two panels)


In [22]:
# =============================================================================
print('\n=== CHART 20 ===')

_perf_l_path = os.path.join('Final', 'NB5', 'model_performance_level.csv')
_perf_d_path = os.path.join('Final', 'NB5', 'model_performance_delta.csv')
_fc_path   = os.path.join('Final', 'NB5', 'ECI_Forecast_2020_2030.csv')
_rank_path  = os.path.join('Final', 'NB5', 'Country_Ranking_2020_2030.csv')
if os.path.exists(_fc_path) and os.path.exists(_rank_path):
    fc   = pd.read_csv(_fc_path)
    rank = pd.read_csv(_rank_path)

    if os.path.exists(_perf_l_path):
        perf_tmp = pd.read_csv(_perf_l_path)
        perf_tmp = perf_tmp[perf_tmp['Model'] != 'XGBoost']
        best_rmse = perf_tmp.iloc[0]['Test RMSE'] if 'Test RMSE' in perf_tmp.columns else 0.08
    else:
        best_rmse = 0.08

    master_tmp = load_master()
    hist = master_tmp[['Country Code', 'Country Name', 'Year',
                        'Economic Complexity Index']].dropna()

    rank_sorted  = rank.sort_values('Total_Change', ascending=False).reset_index(drop=True)
    CASE_STUDIES = ['COG', 'AZE', 'CHL']
    top3    = [cc for cc in rank_sorted['Country Code'].tolist() if cc not in CASE_STUDIES][:3]
    bottom3 = [cc for cc in rank_sorted['Country Code'].tolist()[::-1] if cc not in CASE_STUDIES][:3]

    TOP_COL  = '#2e7d4a'
    BOT_COL  = '#c23a3a'
    GREY_FC  = '#b0b8c4'

    all_eci = pd.concat([
        hist[hist['Country Code'].isin(INCLUDE_LIST)]['Economic Complexity Index'],
        fc[fc['Country Code'].isin(INCLUDE_LIST)]['Ensemble'],
    ]).dropna()
    Y_RANGE = [all_eci.min() - 0.15, all_eci.max() + 0.15]

    def _add_country_traces_fc(fig, cc, cname, col, lw, opacity, legendgroup,
                                row_n, col_n):
        h = hist[hist['Country Code'] == cc].sort_values('Year')
        f = fc[fc['Country Code'] == cc].sort_values('Year')
        if h.empty or f.empty:
            return None
        ens = f['Ensemble'].values
        yrs = f['Year'].values

        fig.add_trace(go.Scatter(
            x=h['Year'], y=h['Economic Complexity Index'],
            mode='lines', line=dict(color=col, width=lw), opacity=opacity,
            legendgroup=legendgroup, showlegend=False,
            hovertemplate=f'<b>{cname} ({cc})</b><br>%{{x}}: %{{y:.3f}}<extra>Historical</extra>',
        ), row=row_n, col=col_n)

        if col != GREY_FC:
            rgb = ','.join(str(v) for v in _hex_to_rgb(col))
            fig.add_trace(go.Scatter(
                x=np.concatenate([yrs, yrs[::-1]]).tolist(),
                y=np.concatenate([ens + best_rmse, (ens - best_rmse)[::-1]]).tolist(),
                fill='toself', fillcolor=f'rgba({rgb},0.08)',
                line=dict(color='rgba(0,0,0,0)'),
                showlegend=False, hoverinfo='skip', legendgroup=legendgroup,
            ), row=row_n, col=col_n)

        last_yr  = int(h['Year'].iloc[-1])
        last_eci = float(h['Economic Complexity Index'].iloc[-1])
        fig.add_trace(go.Scatter(
            x=[last_yr] + yrs.tolist(), y=[last_eci] + ens.tolist(),
            mode='lines', line=dict(color=col, width=lw, dash='dash'),
            opacity=opacity, legendgroup=legendgroup, showlegend=False,
            hovertemplate=f'<b>{cname} ({cc})</b><br>%{{x}}: %{{y:.3f}}<extra>Forecast</extra>',
        ), row=row_n, col=col_n)
        return float(ens[-1])

    def _deconflict_lbl(label_info, min_gap=0.22):
        sorted_lbl = sorted(label_info.items(), key=lambda x: x[1][0])
        adjusted   = {}
        floor_y    = None
        for cc, (y_act, _) in sorted_lbl:
            y_place = y_act if floor_y is None else max(y_act, floor_y + min_gap)
            adjusted[cc] = y_place
            floor_y = y_place
        return adjusted

    fig11 = make_subplots(rows=1, cols=2, horizontal_spacing=0.07)

    for panel_col, highlight_group, highlight_col, grp_name in [
        (1, top3,    TOP_COL, 'top3'),
        (2, bottom3, BOT_COL, 'bottom3'),
    ]:
        fig11.add_vrect(x0=2019.5, x1=2030.5, fillcolor='rgba(200,210,225,0.18)',
                        line=dict(width=0), layer='below', row=1, col=panel_col)
        fig11.add_vline(x=2019.5, line=dict(color='#aaa', width=1.5, dash='dot'),
                        row=1, col=panel_col)

        highlighted_here = set(highlight_group + CASE_STUDIES)

        for cc in INCLUDE_LIST:
            if cc in highlighted_here:
                continue
            row_   = rank[rank['Country Code'] == cc]
            cname_ = row_['Country'].values[0] if len(row_) else cc
            _add_country_traces_fc(fig11, cc, cname_, GREY_FC, lw=0.7, opacity=0.3,
                                   legendgroup='others', row_n=1, col_n=panel_col)

        label_info = {}

        for cc in highlight_group:
            row_   = rank[rank['Country Code'] == cc]
            cname_ = row_['Country'].values[0] if len(row_) else cc
            y_end  = _add_country_traces_fc(fig11, cc, cname_, highlight_col,
                                            lw=2.2, opacity=1.0,
                                            legendgroup=grp_name, row_n=1, col_n=panel_col)
            if y_end is not None:
                label_info[cc] = (y_end, highlight_col)

        adjusted_y = _deconflict_lbl(label_info)
        xref = 'x' if panel_col == 1 else 'x2'
        yref = 'y' if panel_col == 1 else 'y2'

        for cc, (y_act, col) in label_info.items():
            fig11.add_annotation(
                x=2030, y=y_act, ax=2031, ay=adjusted_y[cc],
                axref=xref, ayref=yref, xref=xref, yref=yref,
                text=f'<b>{cc}</b>', showarrow=True,
                arrowhead=2, arrowwidth=1, arrowsize=0.8, arrowcolor=col,
                font=dict(size=9.5, color=col, family=FONT),
                xanchor='left', yanchor='middle',
            )

        fig11.update_xaxes(title_text='Year', gridcolor=GRID, gridwidth=0.5,
                           dtick=5, range=[1994, 2033], row=1, col=panel_col)
        fig11.update_yaxes(
            title_text='Economic Complexity Index' if panel_col == 1 else '',
            range=Y_RANGE, gridcolor=GRID, gridwidth=0.5, row=1, col=panel_col)

    for lbl, col, rk, grp in [
        ('Top 3 improvers — GNQ · MNG · ECU',    TOP_COL,  2, 'top3'),
        ('Bottom 3 decliners — ZWE · SAU · KAZ', BOT_COL,  3, 'bottom3'),
        ('Other countries',                       GREY_FC,  4, 'others'),
    ]:
        fig11.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                                   line=dict(color=col, width=2.5), name=lbl,
                                   legendgroup=grp, showlegend=True, legendrank=rk))
    fig11.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                               line=dict(color='#888', width=1.5), name='── Historical',
                               showlegend=True, legendrank=10))
    fig11.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                               line=dict(color='#888', width=1.5, dash='dash'),
                               name='- - Forecast', showlegend=True, legendrank=11))

    fig11.update_layout(**base_layout(
        height=620, margin=dict(l=70, r=20, t=70, b=110),
        legend=dict(
            orientation='h', font=dict(size=9.5),
            bgcolor='rgba(255,255,255,0.92)', bordercolor=GRID, borderwidth=1,
            x=0.0, y=-0.15, xanchor='left', yanchor='top', tracegroupgap=0,
        ),
    ))
    save(fig11, '20_ml__eci_forecast_top_improvers_2020_2030', OUT_ML, w=1200, h=620)
else:
    print('  SKIPPED: ECI_Forecast or Country_Ranking CSV not found')


=== CHART 20 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/ml/20_ml__eci_forecast_top_improvers_2020_2030.png


## CHART 21 — ECI Forecast heatmap, all countries (historical + projected)


In [23]:
# =============================================================================
print('\n=== CHART 21 ===')

if os.path.exists(_fc_path):
    fc_hm = pd.read_csv(_fc_path)
    master_hm = load_master()
    hist_hm   = (master_hm[master_hm['Country Code'].isin(INCLUDE_LIST)]
                 [['Country Code', 'Country Name', 'Year', 'Economic Complexity Index']]
                 .dropna())

    # Combine historical and forecast
    fc_long = fc_hm[fc_hm['Country Code'].isin(INCLUDE_LIST)][['Country Code', 'Year', 'Ensemble']].copy()
    fc_long = fc_long.rename(columns={'Ensemble': 'ECI'})
    hist_long = hist_hm.rename(columns={'Economic Complexity Index': 'ECI'})
    hist_long = hist_long[['Country Code', 'Year', 'ECI']]

    combined = pd.concat([hist_long, fc_long], ignore_index=True)
    combined = combined.drop_duplicates(subset=['Country Code', 'Year'])

    pivot_hm = combined.pivot(index='Country Code', columns='Year', values='ECI')
    pivot_hm = pivot_hm.reindex(sorted(pivot_hm.index))

    # Sort countries by their 2019 ECI
    sort_col = 2019 if 2019 in pivot_hm.columns else pivot_hm.columns.max()
    pivot_hm = pivot_hm.loc[pivot_hm[sort_col].sort_values().index]

    forecast_start = fc_hm['Year'].min() if 'Year' in fc_hm.columns else 2020

    fig11b = go.Figure(go.Heatmap(
        z=pivot_hm.values,
        x=[str(c) for c in pivot_hm.columns],
        y=pivot_hm.index.tolist(),
        colorscale='RdBu',
        zmid=0,
        colorbar=dict(title='ECI', thickness=16, len=0.9),
        hovertemplate='%{y} | %{x}: %{z:.3f}<extra></extra>',
    ))

    forecast_col_idx = list(pivot_hm.columns).index(forecast_start) if forecast_start in pivot_hm.columns else None
    if forecast_col_idx is not None:
        fig11b.add_shape(type='line',
                         x0=forecast_col_idx - 0.5, x1=forecast_col_idx - 0.5,
                         y0=-0.5, y1=len(pivot_hm) - 0.5,
                         line=dict(color='#333', width=2, dash='dot'),
                         xref='x', yref='y')
        fig11b.add_annotation(
            x=forecast_col_idx - 0.5, y=len(pivot_hm) + 0.2,
            xref='x', yref='y',
            text='<b>← Historical | Forecast →</b>',
            showarrow=False, font=dict(size=10, color='#444'),
            xanchor='center',
        )

    fig11b.update_layout(**base_layout(
        height=max(500, len(pivot_hm) * 12),
        margin=dict(l=80, r=100, t=60, b=80),
        xaxis=dict(tickangle=-45, tickfont=dict(size=9), showgrid=False),
        yaxis=dict(tickfont=dict(size=9), showgrid=False),
    ))
    save(fig11b, '21_ml__eci_forecast_heatmap_all_countries', OUT_ML,
         w=1200, h=max(500, len(pivot_hm) * 12))
else:
    print('  SKIPPED: ECI_Forecast_2020_2030.csv not found')


# =============================================================================


=== CHART 21 ===
  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts/ml/21_ml__eci_forecast_heatmap_all_countries.png


In [24]:
# =============================================================================
# Done
# =============================================================================
print('\n' + '=' * 60)
print('generate_charts.py complete.')
print(f"All outputs written to: {os.path.abspath(os.path.join('Final', 'charts'))}")
print('=' * 60)



generate_charts.py complete.
All outputs written to: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/Final/charts
